# AI-Powered Meta-Software Development Generator

This notebook is an AI-powered meta-software development generator. The user can change the `business_problem` variable and rerun the notebook to generate a different Flask-based software project with SDLC documentation, UML diagrams, a website, a generated image, and a Flask API. The current demonstration business problem is AI Football Match Analyst. DeepSeek API is supported as an optional external AI service, but fallback generation is used by default to ensure reproducibility.

## Phase 1: Inception

### Business problem input

In [ ]:
from pathlib import Path
import os
import json

CURRENT_DIR = Path.cwd()
if CURRENT_DIR.name == 'Task1':
    TASK1_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'Task1').exists():
    TASK1_DIR = CURRENT_DIR / 'Task1'
else:
    TASK1_DIR = CURRENT_DIR

GENERATED_DIR = TASK1_DIR / 'generated_project'
DOCS_DIR = GENERATED_DIR / 'docs'
STATIC_DIR = GENERATED_DIR / 'static'
TEMPLATES_DIR = GENERATED_DIR / 'templates'

for path in [GENERATED_DIR, DOCS_DIR, STATIC_DIR, TEMPLATES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

business_problem = '''
Build an AI Football Match Analyst that supports coaches or analysts by generating
a structured match report from basic match statistics and key events.
'''

USE_EXTERNAL_AI = False

### DeepSeek API configuration

In [ ]:
from dotenv import load_dotenv

load_dotenv()

def call_deepseek(prompt, system_message=None):
    global USE_EXTERNAL_AI
    env_use_external = os.getenv('USE_EXTERNAL_AI', str(USE_EXTERNAL_AI))
    use_external = str(env_use_external).strip().lower() in ('true', '1', 'yes')
    if not use_external:
        return None

    api_key = os.getenv('DEEPSEEK_API_KEY')
    base_url = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
    model = os.getenv('DEEPSEEK_MODEL', 'deepseek-chat')

    if not api_key:
        print('Warning: USE_EXTERNAL_AI is enabled, but DEEPSEEK_API_KEY is missing. Using local fallback.')
        return None

    try:
        import openai
    except ImportError:
        print('Warning: openai package is not installed. Using local fallback.')
        return None

    try:
        client = openai.OpenAI(api_key=api_key, base_url=base_url)
        messages = []
        if system_message:
            messages.append({'role': 'system', 'content': system_message})
        messages.append({'role': 'user', 'content': prompt})
        response = client.chat.completions.create(model=model, messages=messages, temperature=0.7, max_tokens=600)
        return response.choices[0].message.content.strip()
    except Exception as ex:
        print('Warning: DeepSeek API call failed:', str(ex))
        return None

### Generate project specification

In [ ]:
def generate_project_spec(business_problem):
    lower = business_problem.lower()
    if 'football' in lower or 'match' in lower:
        spec = {
            'project_title': 'AI Football Match Analyst',
            'short_description': 'A system that supports coaches or analysts by generating a structured football match report from basic match statistics and key events.',
            'target_users': ['Coach', 'Match Analyst', 'System Reviewer'],
            'user_goals': ['Understand match performance quickly', 'Identify strengths and risks', 'Create coaching recommendations'],
            'input_fields': ['team_a', 'team_b', 'score', 'shots', 'possession', 'events'],
            'workflow_steps': ['Match Input', 'Data Validation', 'AI-style Analysis', 'Report Output', 'Human Review'],
            'api_endpoints': ['GET /', 'GET /api/sample-report', 'POST /api/analyse'],
            'core_entities': ['Match', 'TeamStats', 'Event', 'AIAnalyzer', 'Report', 'FlaskAPI'],
            'ai_integration_point': 'AI-style analysis after data validation.',
            'human_review_point': 'The generated report should be reviewed and edited by a human user before final decision-making.',
            'validation_rules': {'required_fields': ['team_a', 'team_b', 'score', 'shots', 'possession', 'events']}
        }
    else:
        spec = {
            'project_title': 'AI Business Assistant Generator',
            'short_description': business_problem.strip(),
            'target_users': ['Primary User', 'Business Analyst', 'Reviewer'],
            'user_goals': ['Capture inputs', 'Generate insights', 'Review results'],
            'input_fields': ['subject', 'context', 'metrics', 'events'],
            'workflow_steps': ['Input', 'Validate', 'Analyze', 'Report', 'Review'],
            'api_endpoints': ['GET /', 'GET /api/sample-report', 'POST /api/analyse'],
            'core_entities': ['InputData', 'Analyzer', 'Report', 'FlaskAPI'],
            'ai_integration_point': 'AI-assisted analysis after input validation.',
            'human_review_point': 'Users review generated results before taking action.',
            'validation_rules': {'required_fields': ['subject', 'context', 'metrics', 'events']}
        }
    improved = call_deepseek('Generate a project spec for: ' + business_problem)
    if improved:
        spec['deepseek_note'] = improved
    return spec

project_spec = generate_project_spec(business_problem)
print('Project title:', project_spec['project_title'])

### Generate SDLC documentation

In [ ]:
def write_doc(path, content):
    path.write_text(content, encoding='utf-8')
    return str(path)

def generate_problem_statement(spec, business_problem):
    return f'''# Problem Statement

**Background:**
{business_problem.strip()}

**Target Users:**
- {spec['target_users'][0]}
- {spec['target_users'][1]}
- {spec['target_users'][2]}

**Pain Points:**
- Coaches and analysts need fast, structured match summaries.
- Raw statistics are difficult to convert into coaching insights.
- Human review is essential but time-consuming.

**Proposed Solution:**
A Flask-based application that collects match statistics and key events, validates input, uses AI-style analysis to produce a structured report, and surfaces human review notes.'''

def generate_personas(spec):
    return f'''# Personas

## Coach Carla
- Role: Head coach
- Goal: Understand match strengths and weaknesses quickly
- Needs: concise tactical insights with coaching recommendations
- Frustrations: too much raw data and no clear action items

## Analyst Alex
- Role: Match analyst
- Goal: provide structured summaries and evidence-based observations
- Needs: automated report drafts and event-driven insights
- Frustrations: inconsistent analysis quality and repeated manual work

## Reviewer Riley
- Role: Team reviewer
- Goal: verify generated findings and approve final reports
- Needs: clear summaries, human review notes, and reliable validation
- Frustrations: missing context or unverified AI outputs'''

def generate_prd(spec):
    features = '
'.join([f'- {step}' for step in spec['workflow_steps']])
    return f'''# Product Requirements Document

## Overview
A web application that captures input data, validates match statistics, generates an AI-style report, and presents results for human review.

## Main Features
{features}

## User Flow
1. Enter match details and events.
2. Submit to the Flask API.
3. Receive a structured match report.
4. Review and refine the output.

## Constraints
- No external API key is required for core functionality.
- The system should run locally in fallback mode.

## Success Criteria
- Users can submit match data and receive a report.
- The website shows a generated image and report summary.
- Documentation and UML diagrams are produced automatically.'''

def generate_requirements(spec):
    return f'''# Requirements

## Functional Requirements
- Accept match input for: {', '.join(spec['input_fields'])}.
- Validate required fields.
- Generate a structured report with strengths, risks, recommendations, and observations.
- Provide a sample report endpoint.

## Non-functional Requirements
- Local execution without DeepSeek API.
- Clean academic website UI.
- Maintainable code and documentation.

## System Constraints
- Flask web server.
- Optional DeepSeek API integration.

## User Roles
- Coach
- Match Analyst
- System Reviewer'''

def generate_user_stories(spec):
    stories = [
        ('As a coach, I want to submit match statistics so that I can receive a concise performance report.', 'A report is returned with observations and recommendations.'),
        ('As a match analyst, I want the system to identify tactical strengths and risks so that I can prepare notes for the team.', 'The output includes strengths, risks, and tactical commentary.'),
        ('As a reviewer, I want a clear human review note so that I can verify the generated report before final use.', 'The generated response contains a human review note and a flag for review.')
    ]
    items = '
'.join([f'- {story}
  - Acceptance: {criteria}' for story, criteria in stories])
    return f'''# User Stories

{items}'''

def generate_system_design(spec):
    return f'''# System Design

## System Overview
A Flask application that exposes a web UI and API endpoints. It validates input and generates AI-style reports with optional DeepSeek support.

## Workflow
- User submits match data through the website.
- Flask API validates required fields.
- The analyzer generates a report using local logic or DeepSeek.
- The website displays the structured results and review note.

## Flask API Structure
- GET / - renders the website.
- GET /api/sample-report - returns a sample JSON report.
- POST /api/analyse - validates input and returns a generated report.

## AI Integration Point
- The analyzer can use DeepSeek chat completions when configured.

## Data Validation
- Required fields: {', '.join(spec['validation_rules']['required_fields'])}.

## Human Review
- Every report includes a human review note and a review-required flag.

## DeepSeek API Design
- Optional external service with environment variables.
- Safe fallback if the API is unavailable.

## Local Fallback Design
- All functionality works through deterministic logic when no external AI is available.'''

def generate_testing_plan(spec):
    return f'''# Testing Plan

## Unit Testing
- Test Flask routes and validation logic.
- Test report generation with valid and invalid data.

## API Testing
- GET / should return 200.
- GET /api/sample-report should return JSON.
- POST /api/analyse should handle valid payloads and reject missing fields.

## Validation Testing
- Confirm required fields are enforced.

## CI/CD Testing
- Use GitHub Actions to install dependencies and run pytest.

## Deployment Testing
- Ensure the site loads locally.
- Confirm the Flask API returns expected JSON responses.'''

def generate_ai_prompts(spec):
    return f'''# AI Prompts

## Project Specification
Create a project specification for a software system that {business_problem.strip()}.

## Problem Statement
Generate a problem statement describing background, target users, pain points, and a proposed solution.

## Personas
Create at least three personas with goals, needs, and frustrations.

## PRD
Create a product requirements document with overview, features, user flow, constraints, and success criteria.

## User Stories
Create user stories and acceptance criteria for the target users.

## UML Generation
Generate PlantUML source for use case, class, and sequence diagrams based on {spec['project_title']}.

## Flask API Generation
Generate a Flask API with endpoints for `/`, `/api/sample-report`, and `/api/analyse`.

## Website Generation
Generate website text, input fields, and a workflow description for the generated project.

## AI-Style Report Generation
Generate a report prompt that uses match statistics and events to create insights, strengths, risks, and recommendations.'''

docs = {
    'problem_statement.md': generate_problem_statement(project_spec, business_problem),
    'personas.md': generate_personas(project_spec),
    'prd.md': generate_prd(project_spec),
    'requirements.md': generate_requirements(project_spec),
    'user_stories.md': generate_user_stories(project_spec),
    'system_design.md': generate_system_design(project_spec),
    'testing_plan.md': generate_testing_plan(project_spec),
    'ai_prompts.md': generate_ai_prompts(project_spec),
}
for filename, content in docs.items():
    write_doc(DOCS_DIR / filename, content)

print('Generated SDLC documentation files in', DOCS_DIR)

### Generate UML PlantUML files and PNG preview images

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def write_puml(path, text):
    path.write_text(text, encoding='utf-8')

use_cases = '\n'.join([f'  :{actor}: as {actor}' for actor in project_spec['target_users']])
use_cases_text = '\n'.join([
    '  (Submit input data)',
    '  (Validate data)',
    '  (Generate AI-style analysis)',
    '  (Review generated report)',
    '  (View report)'
])
class_lines = '\n'.join([f'  class {name} {{\n    +attributes\n    +methods\n  }}' for name in project_spec['core_entities']])
sequence_lines = '\n'.join([
    'User -> Website: submit match data',
    'Website -> FlaskAPI: POST /api/analyse',
    'FlaskAPI -> Validation: check required fields',
    'FlaskAPI -> AIAnalyzer: generate report',
    'AIAnalyzer -> Report: create structured output',
    'FlaskAPI -> Website: return report JSON',
    'Website -> User: display report and review note'
])
write_puml(DOCS_DIR / 'use_case_diagram.puml', f'@startuml\nleft to right direction\nactor User\n{use_cases_text}\nUser --> (Submit input data)\nUser --> (Review generated report)\nUser --> (View report)\n@enduml')
write_puml(DOCS_DIR / 'class_diagram.puml', f'@startuml\n{class_lines}\nReport <|-- AIAnalyzer\nMatch o-- TeamStats\nMatch o-- Event\n@enduml')
write_puml(DOCS_DIR / 'sequence_diagram.puml', f'@startuml\n{sequence_lines}\n@enduml')

def create_simple_diagram(path, title, blocks):
    width, height = 1000, 600
    image = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((20, 20), title, fill='black', font=font)
    y = 80
    for label, shape in blocks:
        draw.rectangle((40, y, 280, y + 60), outline='black', width=2)
        draw.text((50, y + 20), label, fill='black', font=font)
        if shape == 'arrow':
            draw.line((280, y + 30, 540, y + 30), fill='black', width=2)
            draw.polygon([(540, y + 24), (560, y + 30), (540, y + 36)], fill='black')
            y += 100
        else:
            y += 90
    image.save(path)

create_simple_diagram(DOCS_DIR / 'use_case_diagram.png', 'Use Case Diagram', [
    ('Coach / Analyst', 'box'),
    ('Submit data', 'arrow'),
    ('Receive report', 'box')
])
create_simple_diagram(DOCS_DIR / 'class_diagram.png', 'Class Diagram', [
    ('Match', 'box'),
    ('TeamStats', 'arrow'),
    ('Report', 'box')
])
create_simple_diagram(DOCS_DIR / 'sequence_diagram.png', 'Sequence Diagram', [
    ('User', 'box'),
    ('Website', 'arrow'),
    ('Flask API', 'arrow'),
    ('AI Analyzer', 'arrow'),
    ('Report', 'box')
])

print('Generated UML .puml and PNG files in', DOCS_DIR)

In [ ]:
from IPython.display import display, Image, Markdown

display(Markdown('### UML Diagrams'))
display(Image(filename=str(DOCS_DIR / 'use_case_diagram.png')))
display(Image(filename=str(DOCS_DIR / 'class_diagram.png')))
display(Image(filename=str(DOCS_DIR / 'sequence_diagram.png')))


### Generate Flask API app.py

In [ ]:
app_code = '''from pathlib import Path
import os
from dotenv import load_dotenv
from flask import Flask, jsonify, render_template, request

load_dotenv()

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
DEEPSEEK_MODEL = os.getenv('DEEPSEEK_MODEL', 'deepseek-chat')
USE_EXTERNAL_AI = str(os.getenv('USE_EXTERNAL_AI', 'false')).strip().lower() in ('true', '1', 'yes')

def call_deepseek(prompt, system_message=None):
    if not USE_EXTERNAL_AI:
        return None
    if not DEEPSEEK_API_KEY:
        return None
    try:
        import openai
    except ImportError:
        return None
    try:
        client = openai.OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)
        response = client.chat.completions.create(model=DEEPSEEK_MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.7, max_tokens=600)
        return response.choices[0].message.content.strip()
    except Exception:
        return None

def validate_input(data):
    required = ['team_a', 'team_b', 'score', 'shots', 'possession', 'events']
    missing = [field for field in required if not data.get(field)]
    if missing:
        return False, missing
    return True, []

def interpret_stats(data):
    score = data.get('score', '')
    shots = data.get('shots', '')
    possession = data.get('possession', '')
    events = data.get('events', '')
    summary = []
    summary.append(f'Score reported as {score}.')
    summary.append(f'Possession is {possession}, suggesting control trends.')
    summary.append(f'Shots distribution is {shots}.')
    if 'goal' in events.lower():
        summary.append('Key goals were highlighted in the match events.')
    if 'yellow card' in events.lower() or 'red card' in events.lower():
        summary.append('Discipline and set-piece risk were significant factors.')
    return ' '.join(summary)

def generate_report(data):
    report = {
        'match_summary': interpret_stats(data),
        'strengths': [
            'Strong possession control',
            'Positive attacking momentum',
        ],
        'risks': [
            'Potential defensive lapses under pressure',
            'Discipline risk from cards and set pieces',
        ],
        'recommendations': [
            'Focus on maintaining possession in the final third.',
            'Review defensive transitions and set-piece organization.',
        ],
        'key_observations': [
            'The match showed a good balance of attack and control.',
            'Events suggest moments of high intensity and decision points.',
        ],
        'human_review_note': 'Review the generated report and adjust tactical recommendations to match your team context.',
        'human_review_required': True,
        'success': True
    }
    return report

app = Flask(__name__, template_folder='templates', static_folder='static')

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/api/sample-report')
def sample_report():
    sample = generate_report({
        'team_a': 'Red Lions',
        'team_b': 'Blue Hawks',
        'score': '2-1',
        'shots': '14-10',
        'possession': '58-42',
        'events': 'First-half goal, penalty saved, yellow card'
    })
    return jsonify(sample)

@app.route('/api/analyse', methods=['POST'])
def analyse():
    data = request.get_json(silent=True) or {}
    valid, missing = validate_input(data)
    if not valid:
        return jsonify({'success': False, 'error': 'Missing required fields', 'missing': missing}), 400
    report = generate_report(data)
    external = call_deepseek(f'Generate a football match analysis from: {data}')
    if external:
        report['external_analysis'] = external
    return jsonify(report)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
'''
,
,

: 
,
: {},
: [

: 
,
: null,
: {},
: [],
: [
,
{field}">{field.replace('_', ' ').title()}:</label><input id="{field}" name="{field}" type="text" placeholder="Enter {field.replace('_', ' ') }" required>
' for field in project_spec['input_fields']])
html_template = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>__PROJECT_TITLE__</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 0; padding: 0; background: #f4f6f8; color: #202124; }
    header { background: #0d47a1; color: white; padding: 24px; }
    main { padding: 24px; max-width: 1000px; margin: auto; }
    .card { background: white; border-radius: 12px; box-shadow: 0 2px 12px rgba(0,0,0,0.08); margin-bottom: 20px; padding: 20px; }
    .grid { display: grid; gap: 16px; }
    label { display: block; margin-bottom: 6px; font-weight: 600; }
    input { width: 100%; padding: 10px; margin-bottom: 14px; border: 1px solid #cbd5e0; border-radius: 8px; }
    button { background: #0d47a1; color: white; border: none; padding: 12px 18px; border-radius: 10px; cursor: pointer; }
    button:hover { background: #0b3d91; }
    pre { background: #f1f5f9; padding: 16px; border-radius: 10px; overflow-x: auto; }
    .response-section { white-space: pre-wrap; }
  </style>
</head>
<body>
  <header>
    <h1>__PROJECT_TITLE__</h1>
    <p>__SHORT_DESCRIPTION__</p>
  </header>
  <main>
    <section class="card">
      <img src="/static/generated_image.png" alt="Generated project illustration" style="max-width:100%; border-radius: 12px;">
    </section>
    <section class="card">
      <h2>Workflow</h2>
      <ul>__WORKFLOW_HTML__</ul>
    </section>
    <section class="card">
      <h2>Match Input</h2>
      <form id="analysis-form">
        __FORM_FIELDS_HTML__
        <button type="submit">Generate Report</button>
      </form>
    </section>
    <section class="card">
      <h2>Analysis Result</h2>
      <div id="result">No report generated yet.</div>
    </section>
  </main>
  <script>
    function renderObject(obj) {
      if (typeof obj !== 'object' || obj === null) {
        return document.createTextNode(String(obj));
      }
      const container = document.createElement('div');
      Object.keys(obj).forEach(key => {
        const row = document.createElement('div');
        row.style.marginBottom = '12px';
        const title = document.createElement('strong');
        title.textContent = key + ':
';
        row.appendChild(title);
        const value = obj[key];
        if (typeof value === 'object' && value !== null) {
          row.appendChild(renderObject(value));
        } else {
          const text = document.createElement('div');
          text.textContent = String(value);
          row.appendChild(text);
        }
        container.appendChild(row);
      });
      return container;
    }
    document.getElementById('analysis-form').addEventListener('submit', async event => {
      event.preventDefault();
      const payload = {};
      ['team_a','team_b','score','shots','possession','events'].forEach(field => {
        payload[field] = document.getElementById(field).value.trim();
      });
      const resultEl = document.getElementById('result');
      resultEl.textContent = 'Generating report...';
      const response = await fetch('/api/analyse', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(payload)
      });
      const data = await response.json();
      if (!response.ok) {
        resultEl.textContent = 'Error: ' + (data.error || 'Unknown issue');
        return;
      }
      resultEl.innerHTML = '';
      resultEl.appendChild(renderObject(data));
    });
  </script>
</body>
</html>
'''
,
,
,

: 
,
: {},
: [

: 
,
: null,
: {},
: [],
: [
,
,
,
520
,
,
,
20
20
980
500
,
,
40
,
100
,
180
,
180
30
,
340
,
,

: 
,
: null,
: {},
: [],
: [
,

: 
,
: {},
: [

: 
,
: null,
: {},
: [],
: [
,
,

: 
,
: {},
: [
3